# 🎙️ Fine-tune giọng nói Piper TTS từ dataset của bạn

Notebook này nhận **dataset đã thu bằng VoiceRecorder** (metadata.csv + các file .wav), fine-tune từ checkpoint tiếng Việt có sẵn (`vi_VN-vais1000-medium`), rồi xuất ra `.onnx` + `.onnx.json` để dùng với Piper.

**Trước khi chạy:**
1. Vào **Runtime → Change runtime type → GPU (T4)**.
2. Nén thư mục `dataset` (nằm cạnh file `.exe` đã build, ví dụ `...\bin\Release\net9.0-windows\dataset\`) thành `dataset.zip`.
3. Chạy **lần lượt từng ô từ trên xuống dưới** (bấm ▶), hoặc **Runtime → Run all**. Không bỏ qua/nhảy cóc ô nào — mỗi lần mở notebook mới hoặc restart runtime, mọi thứ (đã mount Drive, đã giải nén...) đều mất sạch và phải chạy lại từ đầu.

**Nếu Google Drive không kết nối được (hay gặp trên điện thoại):** ở Bước 0 tắt công tắc `USE_DRIVE`, notebook sẽ chuyển sang chế độ **upload file trực tiếp** — không cần đăng nhập Drive. Xem ghi chú rủi ro ở Bước 0.

**Lưu ý về license:** notebook này dùng bộ code huấn luyện gốc của `rhasspy/piper` (MIT) vì nó tương thích trực tiếp với checkpoint `vais1000-medium`. Bản kế nhiệm `OHF-Voice/piper1-gpl` (GPL-3.0) hiện cộng đồng báo là chưa nạp ổn định các checkpoint cũ.

## Bước 0: Thiết lập tham số

In [ ]:
#@markdown ### Nguồn dữ liệu
#@markdown Bật nếu muốn dùng Google Drive (khuyến nghị — giữ được tiến độ train nếu Colab bị ngắt). Tắt nếu Drive không kết nối được, sẽ chuyển sang upload file trực tiếp từ máy/điện thoại.
USE_DRIVE = True  #@param {type:"boolean"}

#@markdown Đường dẫn tới `dataset.zip` trên Drive (chỉ dùng khi USE_DRIVE = True):
DATASET_ZIP = "/content/drive/MyDrive/dataset.zip"  #@param {type:"string"}

#@markdown Tên giọng nói (chỉ chữ/số/gạch dưới, không dấu, không khoảng trắng):
VOICE_NAME = "giong_cua_toi"  #@param {type:"string"}

#@markdown Thư mục trên Drive để lưu tiến độ train (chỉ dùng khi USE_DRIVE = True):
DRIVE_ROOT = "/content/drive/MyDrive/piper_training"  #@param {type:"string"}

#@markdown Batch size (Colab T4 free thường hợp 8-16; giảm xuống nếu bị lỗi hết bộ nhớ GPU):
BATCH_SIZE = 12  #@param {type:"integer"}

#@markdown Train thêm bao nhiêu epoch nữa tính từ checkpoint hiện tại:
EXTRA_EPOCHS = 1000  #@param {type:"integer"}

if not USE_DRIVE:
    print('CANH BAO: dang chay o CHE DO KHONG DUNG DRIVE.')
    print('Tien do train (checkpoint) se luu tam trong phien Colab (/content) va SE MAT khi Colab ngat ket noi hoac het gio.')
    print('Chi nen dung che do nay de test nhanh voi it du lieu. Khi train that (nhieu gio), nen tim cach mount duoc Drive (vi du mo notebook nay bang Chrome tren may tinh).')

## Bước 1: Kết nối dữ liệu
Tự động dùng Drive hoặc upload trực tiếp tuỳ theo `USE_DRIVE` ở Bước 0.

In [ ]:
import os

if USE_DRIVE:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=True)
    except Exception as e:
        print('MOUNT DRIVE THAT BAI:', e)
        print('Hay quay lai Buoc 0, tat USE_DRIVE, roi chay lai tu Buoc 0.')
        raise

    assert os.path.exists(DATASET_ZIP), f'Khong tim thay file: {DATASET_ZIP}. Kiem tra lai duong dan tren Drive.'
    dataset_zip_path = DATASET_ZIP
    TRAIN_ROOT = DRIVE_ROOT
else:
    from google.colab import files
    print('Chon file dataset.zip tu may/dien thoai:')
    uploaded = files.upload()
    assert uploaded, 'Ban chua chon file nao.'
    dataset_zip_path = '/content/' + list(uploaded.keys())[0]
    TRAIN_ROOT = '/content/piper_training'

os.makedirs('/content/dataset_raw', exist_ok=True)
!unzip -q -o "{dataset_zip_path}" -d /content/dataset_raw
os.makedirs(TRAIN_ROOT, exist_ok=True)

print('Da giai nen xong. Noi dung:')
!find /content/dataset_raw -maxdepth 4 -type d
print()
print('So file tim thay:')
!find /content/dataset_raw -type f | wc -l

## Bước 1b: Gộp các bộ câu (nếu bạn thu bằng nhiều "Kịch bản" khác nhau) thành 1 dataset duy nhất

VoiceRecorder lưu mỗi bộ câu (kịch bản) vào một thư mục con riêng (`<ten_bo_cau>/wavs` + `<ten_bo_cau>/metadata.csv`), mỗi bộ đều đánh số file lại từ `0001` — nên không thể ghép trực tiếp, phải đổi số lại cho không trùng. Ô dưới tự dò và gộp hết lại.

In [ ]:
import shutil

combined_wavs = '/content/combined_dataset/wavs'
combined_meta = '/content/combined_dataset/metadata.csv'
os.makedirs(combined_wavs, exist_ok=True)

# tim tat ca cac thu muc co ca wavs/ va metadata.csv ben trong (moi thu muc la 1 bo cau)
dataset_sets = []
for root, dirs, filenames in os.walk('/content/dataset_raw'):
    if 'metadata.csv' in filenames and 'wavs' in dirs:
        dataset_sets.append(root)

if len(dataset_sets) == 0:
    print('KHONG TIM THAY BO DU LIEU NAO. Toan bo file da giai nen:')
    !find /content/dataset_raw -type f
    print()
    print('=> Kiem tra: (1) dataset.zip co dung la nen thu muc "dataset" canh file .exe khong,')
    print('   (2) trong app VoiceRecorder dong chu "Da thu: X" co X > 0 khong (X=0 nghia la chua luu duoc cau nao, se khong co metadata.csv).')

assert len(dataset_sets) > 0, 'Khong tim thay bo du lieu nao (can co wavs/ va metadata.csv). Xem log phia tren de biet ly do.'
print(f'Tim thay {len(dataset_sets)} bo cau:')
for s in dataset_sets:
    print(' -', s)

global_id = 1
rows = []
for set_dir in sorted(dataset_sets):
    meta_path = os.path.join(set_dir, 'metadata.csv')
    wavs_dir = os.path.join(set_dir, 'wavs')
    with open(meta_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or '|' not in line:
                continue
            old_id, text = line.split('|', 1)
            src_wav = os.path.join(wavs_dir, old_id.strip() + '.wav')
            if not os.path.exists(src_wav):
                continue
            new_id = f'{global_id:04d}'
            shutil.copy(src_wav, os.path.join(combined_wavs, new_id + '.wav'))
            rows.append(f'{new_id}|{text}')
            global_id += 1

assert len(rows) > 0, 'Tim thay thu muc bo cau nhung metadata.csv rong hoac khong khop voi file .wav nao ca.'

with open(combined_meta, 'w', encoding='utf-8') as f:
    f.write('\n'.join(rows) + '\n')

print(f'Da gop xong: {len(rows)} cau thoai vao /content/combined_dataset')

## Bước 2: Cài đặt môi trường huấn luyện Piper
(Chạy 1 lần, mất khoảng 3-5 phút)

In [ ]:
!apt-get -qq install -y espeak-ng > /dev/null
!git clone -q https://github.com/rhasspy/piper.git /content/piper

# requirements.txt goc cua piper ghim nhieu phien ban qua cu, khong con phu hop voi
# moi truong Colab hien tai (Python 3.12 + torch 2.11 co san). Vá lai truoc khi cai:
# 1. piper-phonemize~=1.1.0  -> dung ban thay the 'piper-phonemize-fix'
# 2. pytorch-lightning~=1.7.0 -> ghim sang 1.9.5 (moi hon nhung van gan API cu)
# 3. torch<2,>=1.11.0        -> bo han rang buoc, dung luon torch co san tren Colab
# 4. cython>=0.29.0,<1       -> ghim sang Cython>=3 de tuong thich Python 3.12
!pip install -q piper-phonemize-fix

import re
req_path = '/content/piper/src/python/requirements.txt'
setup_path = '/content/piper/src/python/setup.py'

with open(req_path, encoding='utf-8') as f:
    content = f.read()
content = re.sub(r'(?im)^.*piper-phonemize.*\n?', '', content)
content = re.sub(r'(?im)^pytorch-lightning.*\n?', 'pytorch-lightning==1.9.5\n', content)
content = re.sub(r'(?im)^torch\s*[<>=!~].*\n?', '', content)
content = re.sub(r'(?im)^torchvision\s*[<>=!~].*\n?', '', content)
content = re.sub(r'(?im)^torchaudio\s*[<>=!~].*\n?', '', content)
content = re.sub(r'(?im)^cython.*\n?', 'cython>=3.0.0\n', content)
with open(req_path, 'w', encoding='utf-8') as f:
    f.write(content)
print('Da sua requirements.txt xong.')

with open(setup_path, encoding='utf-8') as f:
    setup_content = f.read()
patched_setup = re.sub(r'(?im)^.*piper-phonemize.*\n?', '', setup_content)
if patched_setup != setup_content:
    with open(setup_path, 'w', encoding='utf-8') as f:
        f.write(patched_setup)
    print('Da go dong piper-phonemize trong setup.py')

!pip install -q -U 'cython>=3.0.0'

%cd /content/piper/src/python
!pip install -q -e .
!rm -f /content/piper/src/python/piper_train/vits/monotonic_align/core.c
!bash build_monotonic_align.sh

ok = True
try:
    import piper_phonemize
    print('piper_phonemize hoat dong binh thuong.')
except ImportError as e:
    ok = False
    print('LOI: piper_phonemize chua import duoc:', e)

try:
    import torch
    import pytorch_lightning
    import piper_train
    print('torch', torch.__version__, '| pytorch_lightning', pytorch_lightning.__version__, '| piper_train da cai dat thanh cong.')
    print('GPU kha dung:', torch.cuda.is_available())
except ImportError as e:
    ok = False
    print('LOI: chua cai dat duoc:', e)

import glob
so_files = glob.glob('/content/piper/src/python/piper_train/vits/monotonic_align/**/*.so', recursive=True)
print('File .so tim thay:', so_files if so_files else '(khong co)')
monotonic_align_ok = len(so_files) > 0

# kiem tra thuc te bang cach import module dung nhu code piper se import
try:
    from piper_train.vits import monotonic_align
    print('Import piper_train.vits.monotonic_align: OK')
    monotonic_align_ok = True
except Exception as e:
    print('Import piper_train.vits.monotonic_align: LOI ->', e)
    monotonic_align_ok = False

ok = ok and monotonic_align_ok
print('Cai dat xong.' if ok else 'CAI DAT CHUA HOAN TAT - xem loi phia tren.')

## Bước 2b: Vá lỗi "Weights only load failed" (torch 2.6+)
Từ PyTorch 2.6, `torch.load` mặc định bật `weights_only=True`, khiến các checkpoint cũ (như `vais1000-medium`) load bị lỗi `UnpicklingError`. Ô dưới vá lại `lightning_fabric` để luôn load với `weights_only=False` (an toàn vì checkpoint lấy từ kho chính thức `rhasspy/piper-checkpoints`).

In [ ]:
import re
import lightning_fabric.utilities.cloud_io as cloud_io

cloud_io_path = cloud_io.__file__
with open(cloud_io_path, encoding="utf-8") as f:
    content = f.read()

patched = re.sub(
    r"torch\.load\(f, map_location=map_location\)",
    "torch.load(f, map_location=map_location, weights_only=False)",
    content,
)

if patched != content:
    with open(cloud_io_path, "w", encoding="utf-8") as f:
        f.write(patched)
    print("Da va xong:", cloud_io_path)
else:
    if "weights_only=False" in content:
        print("Da duoc va tu truoc do, khong can lam gi them.")
    else:
        print("KHONG TIM THAY DONG CAN VA. Duong dan file:", cloud_io_path)
        print("Hay bao loi nay lai, co the phien ban pytorch_lightning da doi cach load checkpoint.")

## Bước 3: Tiền xử lý dataset
Chuẩn hoá audio (resample 22050Hz, mono) và sinh cache phoneme từ `combined_dataset`.

In [ ]:
PREPROCESS_DIR = f'{TRAIN_ROOT}/{VOICE_NAME}_train'
os.makedirs(PREPROCESS_DIR, exist_ok=True)

%cd /content/piper/src/python
!python3 -m piper_train.preprocess \
  --language vi \
  --input-dir /content/combined_dataset \
  --output-dir "{PREPROCESS_DIR}" \
  --dataset-format ljspeech \
  --single-speaker \
  --sample-rate 22050

print('Preprocess xong, du lieu luu tai:', PREPROCESS_DIR)

## Bước 4: Tải checkpoint tiếng Việt `vais1000-medium` để fine-tune
(Tự tìm đúng tên file checkpoint trên Hugging Face, không cần tự tra tên file)

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import HfApi, hf_hub_download
import re

REPO_ID = 'rhasspy/piper-checkpoints'
CKPT_PREFIX = 'vi/vi_VN/vais1000/medium/'

api = HfApi()
files_list = api.list_repo_files(REPO_ID, repo_type='dataset')
ckpt_candidates = [f for f in files_list if f.startswith(CKPT_PREFIX) and f.endswith('.ckpt')]
assert ckpt_candidates, 'Khong tim thay checkpoint vais1000-medium tren Hugging Face.'
ckpt_remote_path = ckpt_candidates[0]
print('Checkpoint:', ckpt_remote_path)

pretrained_ckpt = hf_hub_download(repo_id=REPO_ID, filename=ckpt_remote_path, repo_type='dataset')
print('Da tai ve:', pretrained_ckpt)

m = re.search(r'epoch=(\d+)', ckpt_remote_path)
pretrained_epoch = int(m.group(1)) if m else 0
print('Epoch cua checkpoint goc:', pretrained_epoch)

## Bước 5: Fine-tune
Ô này tự phát hiện nếu bạn đã train dở từ trước (checkpoint đã lưu trong `TRAIN_ROOT`) để train tiếp; nếu chưa thì bắt đầu từ checkpoint `vais1000-medium` gốc.

In [ ]:
import glob

TRAINING_DIR = PREPROCESS_DIR  # piper luu lightning_logs/checkpoint ngay trong day
existing_ckpts = glob.glob(f'{TRAINING_DIR}/lightning_logs/version_*/checkpoints/*.ckpt')

if existing_ckpts:
    existing_ckpts.sort(key=os.path.getmtime)
    resume_ckpt = existing_ckpts[-1]
    m = re.search(r'epoch=(\d+)', os.path.basename(resume_ckpt))
    start_epoch = int(m.group(1)) if m else pretrained_epoch
    print('Phat hien tien do da train truoc do, tiep tuc tu:', resume_ckpt)
else:
    resume_ckpt = pretrained_ckpt
    start_epoch = pretrained_epoch
    print('Bat dau fine-tune moi tu checkpoint vais1000-medium goc.')

max_epochs = start_epoch + EXTRA_EPOCHS
print(f'Se train den epoch {max_epochs} (hien tai dang o epoch {start_epoch}).')

In [ ]:
%cd /content/piper/src/python
!python3 -m piper_train \
  --dataset-dir "{TRAINING_DIR}" \
  --accelerator gpu \
  --devices 1 \
  --batch-size {BATCH_SIZE} \
  --validation-split 0.0 \
  --num-test-examples 0 \
  --quality medium \
  --checkpoint-epochs 1 \
  --precision 32 \
  --max_epochs {max_epochs} \
  --resume_from_checkpoint "{resume_ckpt}"

## Bước 5b: Dọn checkpoint cũ (tiết kiệm dung lượng Drive)
Mỗi lần restart, Lightning tạo thư mục `version_x` mới và checkpoint chất chồng qua các thư mục. Ô này gom tất cả checkpoint trong mọi `version_*`, chỉ giữ lại **N checkpoint có epoch mới nhất**, xoá phần còn lại để đỡ tốn dung lượng Drive (mỗi file ~800MB+). Chạy ô này sau khi train xong (hoặc bất cứ lúc nào muốn dọn), **không ảnh hưởng** tới việc resume vì Bước 5 luôn tự tìm checkpoint epoch cao nhất.

In [ ]:
import glob, os, re

KEEP_LATEST = 3  # so luong checkpoint moi nhat muon giu lai

all_ckpts = glob.glob(f"{TRAINING_DIR}/lightning_logs/version_*/checkpoints/*.ckpt")

def _epoch_of(path):
    m = re.search(r"epoch=(\d+)", os.path.basename(path))
    return int(m.group(1)) if m else -1

all_ckpts.sort(key=_epoch_of)

if len(all_ckpts) <= KEEP_LATEST:
    print(f"Chi co {len(all_ckpts)} checkpoint, chua can don dep.")
else:
    to_delete = all_ckpts[:-KEEP_LATEST]
    to_keep = all_ckpts[-KEEP_LATEST:]
    freed = 0
    for p in to_delete:
        freed += os.path.getsize(p)
        os.remove(p)
    print(f"Da xoa {len(to_delete)} checkpoint cu, giai phong {freed / (1024**3):.2f} GB.")
    print("Con lai (giu lai):")
    for p in to_keep:
        print(" -", p)

**Nếu Colab bị ngắt giữa lúc train:** mở lại notebook, chạy lại từ Bước 0 (nhập đúng lại `VOICE_NAME` và `DRIVE_ROOT`/`USE_DRIVE` như cũ) đến hết Bước 2, rồi chạy tiếp từ Bước 5 — không cần chạy lại Bước 1/1b/3 nếu dữ liệu tiền xử lý đã nằm sẵn trong Drive (`USE_DRIVE = True`). Nếu đang chạy `USE_DRIVE = False`, dữ liệu đã mất, phải làm lại từ Bước 1.

**Muốn train thêm nữa sau khi đã xong?** Chỉ cần tăng `EXTRA_EPOCHS` ở Bước 0 rồi chạy lại 2 ô của Bước 5.

## Bước 5c: Chuyển sang tài khoản Google khác khi hết GPU quota
Colab giới hạn GPU theo **tài khoản Google**, còn dữ liệu/checkpoint nằm ở Drive nên tách biệt được. Quy trình:

1. Chạy ô **đóng gói** dưới đây (ở tài khoản đang hết quota) để nén cả thư mục train (dataset đã tiền xử lý + checkpoint mới nhất) thành 1 file `.zip` trong Drive.
2. Chuyển file zip sang Drive của tài khoản mới. Cách nhanh nhất (không tốn băng thông của bạn): mở Google Drive trên web, **chia sẻ (Share)** file zip đó cho email tài khoản mới, rồi ở tài khoản mới bấm **"Add shortcut to Drive" → "Copy to My Drive"**. Cách này Google copy nội bộ, nhanh hơn nhiều so với tải về máy rồi upload lại.
3. Ở Colab của tài khoản mới: mount Drive, chạy ô **giải nén** dưới đây để đưa dữ liệu về đúng cấu trúc thư mục `piper_training/<VOICE_NAME>_train`.
4. Chạy lại Bước 0 (nhập đúng `VOICE_NAME` như cũ) → Bước 2 → Bước 2b (cài môi trường), **bỏ qua Bước 1/1b/3**, rồi chạy thẳng Bước 5 — code sẽ tự tìm checkpoint epoch cao nhất và train tiếp bình thường.

In [ ]:
# Chay o TAI KHOAN DANG HET QUOTA de dong goi du lieu
import shutil, os

ZIP_BASENAME = f"{TRAIN_ROOT}/{VOICE_NAME}_train_backup"
zip_path = shutil.make_archive(ZIP_BASENAME, "zip", TRAINING_DIR)

size_gb = os.path.getsize(zip_path) / (1024 ** 3)
print("Da dong goi xong:", zip_path)
print(f"Kich thuoc: {size_gb:.2f} GB")
print("Buoc tiep theo: chia se file zip nay sang tai khoan Google khac qua Drive web.")

In [ ]:
# Chay o TAI KHOAN MOI, SAU KHI da copy file zip vao Drive tai khoan nay
import zipfile, os

# Sua duong dan nay cho dung noi ban vua copy file zip vao Drive tai khoan moi
ZIP_PATH = f"{TRAIN_ROOT}/{VOICE_NAME}_train_backup.zip"

EXTRACT_DIR = f"{TRAIN_ROOT}/{VOICE_NAME}_train"
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

print("Da giai nen xong vao:", EXTRACT_DIR)
print("Gio co the chay lai tu Buoc 0 -> Buoc 2b roi vao thang Buoc 5.")

## Bước 6: Xuất ra ONNX và nghe thử

In [ ]:
ckpts = glob.glob(f'{TRAINING_DIR}/lightning_logs/version_*/checkpoints/*.ckpt')
ckpts.sort(key=os.path.getmtime)
final_ckpt = ckpts[-1]
print('Dung checkpoint moi nhat:', final_ckpt)

onnx_path = f'/content/{VOICE_NAME}.onnx'
%cd /content/piper/src/python
!python3 -m piper_train.export_onnx "{final_ckpt}" "{onnx_path}"

config_path = f'{onnx_path}.json'
shutil.copy(f'{TRAINING_DIR}/config.json', config_path)
print('Da xuat xong:', onnx_path, 'va', config_path)

In [ ]:
!pip install -q piper-tts

test_text = 'Xin chào, đây là giọng nói được huấn luyện từ dữ liệu của bạn.'  #@param {type:"string"}
!echo "{test_text}" | piper --model "{onnx_path}" --output_file /content/test_output.wav

from IPython.display import Audio, display
display(Audio('/content/test_output.wav'))

## Bước 7: Lưu file cuối cùng

In [ ]:
if USE_DRIVE:
    FINAL_DIR = f'{DRIVE_ROOT}/{VOICE_NAME}_ket_qua'
    os.makedirs(FINAL_DIR, exist_ok=True)
    shutil.copy(onnx_path, f'{FINAL_DIR}/{VOICE_NAME}.onnx')
    shutil.copy(config_path, f'{FINAL_DIR}/{VOICE_NAME}.onnx.json')
    print('Da luu vao Drive:', FINAL_DIR)
else:
    from google.colab import files
    print('Dang tai file ve may/dien thoai...')
    files.download(onnx_path)
    files.download(config_path)

print('2 file .onnx + .onnx.json nay dung truc tiep voi app .NET (piper voice).')

## 🔁 Train theo nhiều đợt (thu thêm dữ liệu rồi train tiếp)

Bạn hoàn toàn có thể không cần thu hết một lần — thu một phần, train thử, rồi quay lại thu thêm và train tiếp:

1. Thu một phần câu bằng VoiceRecorder (ví dụ 100-150 câu đầu).
2. Nén `dataset` → `dataset.zip`, chạy notebook này từ đầu (đợt train 1).
3. Thu thêm câu (không quan trọng thu bộ cũ hay bộ mới).
4. Nén lại toàn bộ thư mục `dataset` đè lên `dataset.zip` cũ trên Drive.
5. Mở lại notebook, giữ nguyên `VOICE_NAME`, `USE_DRIVE`, `DRIVE_ROOT` như đợt trước, chạy lại từ Bước 1 đến Bước 5.
6. Bước 5 sẽ tự nhận ra checkpoint đã train từ đợt trước và train tiếp — không mất tiến độ cũ.

Lặp lại bước 3-6 bao nhiêu đợt tuỳ ý. Chỉ cần giữ tên `VOICE_NAME`, `USE_DRIVE`, `DRIVE_ROOT` không đổi giữa các đợt — và bắt buộc phải dùng `USE_DRIVE = True` để cách này hoạt động (chế độ không Drive không giữ được gì giữa các phiên).

## Ghi chú / xử lý sự cố thường gặp

- **`CUDA out of memory`**: giảm `BATCH_SIZE` ở Bước 0 (thử 8, rồi 4), chạy lại Bước 5.
- **Train xong nhưng giọng nghe chưa giống / còn lỗi phát âm**: tăng `EXTRA_EPOCHS` rồi chạy lại Bước 5 và Bước 6 — không cần làm lại từ đầu.
- **Câu quá dài bị bỏ qua khi train**: thêm `--max-phoneme-ids 400` vào lệnh train ở Bước 5 nếu log báo drop câu.
- **`AssertionError: Khong tim thay bo du lieu nao` ở Bước 1b**: xem log ngay phía trên assert đó — nó liệt kê toàn bộ file đã giải nén được. Hai nguyên nhân hay gặp nhất: (1) `dataset.zip` được nén từ thư mục project thay vì thư mục `dataset` cạnh file `.exe` đã build, (2) chưa thu được câu nào thành công trong app (kiểm tra dòng "Đã thu: X" > 0 trong VoiceRecorder trước khi nén).
- **Mount Drive báo lỗi (nút ▶ hiện dấu đỏ) dù đã cấp quyền**: hay gặp trên trình duyệt điện thoại — do mở Colab qua webview của app khác (Zalo/Messenger...) thay vì Chrome thật, hoặc Chrome đang bật "Lite mode"/tiết kiệm dữ liệu. Thử: mở đúng bằng Chrome, tắt Lite mode, hoặc đơn giản nhất — quay lại Bước 0 tắt `USE_DRIVE` để chuyển sang upload file trực tiếp, không cần Drive.
- **Mở lại notebook / bấm Restart runtime**: mọi thứ trong phiên cũ (đã mount Drive, đã giải nén, biến đã gán...) đều mất sạch. Luôn chạy lại **từ Bước 0** theo đúng thứ tự, không nhảy cóc — dễ nhất là dùng `Runtime → Run all`.
- Notebook cố tình dùng repo `rhasspy/piper` (đã archived nhưng vẫn chạy tốt) thay vì `OHF-Voice/piper1-gpl` mới hơn, vì mới hơn hiện chưa nạp ổn định checkpoint `vais1000-medium` cũ.